In [1]:
import ast
import numpy as np
import pandas as pd
import pydicom
from glob import glob

# 1) read CSV
df = pd.read_csv("train_localizers.csv")

# 2) load dicoms for one series and sort
dicom_files = glob("/Users/nicolas/Desktop/tochange/*.dcm")
ds_list = [pydicom.dcmread(f, stop_before_pixels=True) for f in dicom_files] # Reads the metadata of all dicom files

In [2]:
# robust sort (prefer ImagePositionPatient, fallback InstanceNumber)
def sort_key(ds):
    if hasattr(ds, "ImagePositionPatient"):
        return float(ds.ImagePositionPatient[2])
    return int(ds.InstanceNumber)

ds_list.sort(key=sort_key)

In [3]:
sop_to_z = {ds.SOPInstanceUID: i for i, ds in enumerate(ds_list)}

In [4]:
# Get the In-plane spacing (row, col) => dy, dx
dy, dx = map(float, ds_list[0].PixelSpacing) # Get the physical space between pixel centers in mm/pixel

# Slice spacing: Prefer slice deltas if available (ImagePositionPatient)

if hasattr(ds_list[0], "SpacingBetweenSlices"):
    dz = float(ds_list[0].SpacingBetweenSlices)
else:
    dz = float(getattr(ds_list[0], "SliceThickness", 1.0))

In [5]:
Z = len(ds_list)
Y = int(ds_list[0].Rows)
X = int(ds_list[0].Columns)
mask = np.zeros((Z, Y, X), dtype=np.uint8)

In [6]:
# Paint sphere:

def paint_sphere_mm(mask3d, cz, cy, cx, r_mm, dz, dy, dx, value=1):
    # convert physical radius to voxel extents
    rz = int(np.ceil(r_mm / dz))
    ry = int(np.ceil(r_mm / dy))
    rx = int(np.ceil(r_mm / dx))

    z0 = max(0, cz - rz)
    z1 = min(mask3d.shape[0] - 1, cz + rz)
    y0 = max(0, cy - ry)
    y1 = min(mask3d.shape[1] - 1, cy + ry)
    x0 = max(0, cx - rx)
    x1 = min(mask3d.shape[2] - 1, cx + rx)

    zz, yy, xx = np.ogrid[z0:z1+1, y0:y1+1, x0:x1+1]

    # squared distance in mm
    dist2 = ((zz - cz) * dz)**2 + ((yy - cy) * dy)**2 + ((xx - cx) * dx)**2
    sph = dist2 <= (r_mm * r_mm)

    mask3d[z0:z1+1, y0:y1+1, x0:x1+1][sph] = value    


In [7]:
R_mm = 1.5  # radius in millimeters
for _, row in df.iterrows():
    sop = row["SOPInstanceUID"]
    if sop not in sop_to_z:
        continue

    xy = ast.literal_eval(row["coordinates"])
    x = int(round(xy["x"]))
    y = int(round(xy["y"]))
    z = sop_to_z[sop]

    if 0 <= x < X and 0 <= y < Y and 0 <= z < Z:
        paint_sphere_mm(mask, z, y, x, R_mm, dz, dy, dx, value=1)

In [8]:
# Save it however you save your NIfTI (using nibabel) or keep as numpy.

import os
import numpy as np
import nibabel as nib
import dicom2nifti

# --- your code above builds: ds_list (sorted) and mask (Z,Y,X) ---

# 1) Convert the DICOM series to a reference NIfTI (to get affine/header)
dicom_dir = "/Users/nicolas/Desktop/tochange"
ref_nii_path = "/Users/nicolas/Desktop/ref_image.nii.gz"
dicom2nifti.dicom_series_to_nifti(dicom_dir, ref_nii_path, reorient_nifti=True)

ref_img = nib.load(ref_nii_path)
ref_affine = ref_img.affine
ref_header = ref_img.header.copy()

# 2) Make sure the mask axis order matches the reference image axis order
# dicom2nifti (with reorient=True) typically returns data shaped (X, Y, Z)
# while your mask is (Z, Y, X). So transpose:
mask_xyz = np.transpose(mask, (2, 1, 0))  # (Z,Y,X) -> (X,Y,Z)

# If shapes don't match, you need to fix ordering/reorient choice
print("ref shape:", ref_img.shape, "mask shape:", mask_xyz.shape)

# 3) Save mask as NIfTI (uint8 labels)
mask_img = nib.Nifti1Image(mask_xyz.astype(np.uint8), ref_affine, header=ref_header)
mask_img.set_data_dtype(np.uint8)

out_path = "/Users/nicolas/Desktop/mask1.nii.gz"
nib.save(mask_img, out_path)
print("Saved:", out_path)

ref shape: (512, 512, 671) mask shape: (512, 512, 671)
Saved: /Users/nicolas/Desktop/mask1.nii.gz


In [20]:
df.groupby("SOPInstanceUID").count()

,SeriesInstanceUID,coordinates,location
SOPInstanceUID,,,
1.2.826.0.1.3680043.8.498.10004954834025258704096974092309573184,1,1,1
1.2.826.0.1.3680043.8.498.10008247548531974669766933171663663227,1,1,1
1.2.826.0.1.3680043.8.498.10016691921126212638179847078125865225,1,1,1
1.2.826.0.1.3680043.8.498.10017139921419623156923393828313825765,1,1,1
1.2.826.0.1.3680043.8.498.10018298782115251227160833506685050077,1,1,1
...,...,...,...
1.2.826.0.1.3680043.8.498.99755045998570323260194356973585168539,1,1,1
1.2.826.0.1.3680043.8.498.99826270310316038601146280775892979877,1,1,1
1.2.826.0.1.3680043.8.498.99831194919775054219967834262365569056,1,1,1
